# 02 - Storage / DocumentStore Service Test

هذا الـ Notebook يختبر خدمة التخزين فقط.


In [ ]:
from pathlib import Path
import sys, os, json, sqlite3, subprocess, time
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# يستطيع صديقك تغيير هذا المتغير إذا كانت artifacts خارج المشروع.
# مثال في PowerShell قبل فتح notebook:
# $env:IR_ARTIFACT_ROOT="E:\\ir_project_artifacts"
ARTIFACT_ROOT = Path(os.environ.get("IR_ARTIFACT_ROOT", r"E:\ir_project_artifacts"))


def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return Path(paths[0])

DB_PATH = first_existing([
    PROJECT_ROOT / "data" / "documents.sqlite",
    PROJECT_ROOT / "data" / "documents.db",
    ARTIFACT_ROOT / "documents.sqlite",
    ARTIFACT_ROOT / "documents.db",
])

TERRIER_INDEX_PATH = first_existing([
    PROJECT_ROOT / "indexes" / "terrier_medline",
    PROJECT_ROOT / "data" / "indexes" / "terrier_medline",
    ARTIFACT_ROOT / "indexes" / "terrier_medline",
])

BERT_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_bert",
    PROJECT_ROOT / "indexes" / "faiss_bert_full",
    PROJECT_ROOT / "data" / "rag_artifacts",
    ARTIFACT_ROOT / "indexes" / "faiss_bert_full",
    ARTIFACT_ROOT / "indexes" / "faiss_bert",
])

WORD2VEC_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_word2vec",
    PROJECT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec",
])

REPORTS_DIR = PROJECT_ROOT / "reports" / "evaluation_notebook"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT       =", PROJECT_ROOT)
print("ARTIFACT_ROOT      =", ARTIFACT_ROOT)
print("DB_PATH            =", DB_PATH, "exists=", DB_PATH.exists())
print("TERRIER_INDEX_PATH =", TERRIER_INDEX_PATH, "exists=", TERRIER_INDEX_PATH.exists())
print("BERT_INDEX_DIR     =", BERT_INDEX_DIR, "exists=", BERT_INDEX_DIR.exists())
print("WORD2VEC_INDEX_DIR =", WORD2VEC_INDEX_DIR, "exists=", WORD2VEC_INDEX_DIR.exists())

In [ ]:
assert DB_PATH.exists(), f"SQLite not found: {DB_PATH}"

with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
    schema = pd.read_sql_query("PRAGMA table_info(documents)", conn)
    count_df = pd.read_sql_query("SELECT COUNT(*) AS documents_count FROM documents", conn)

print("SQLite file:", DB_PATH)
display(tables)
display(schema)
display(count_df)

In [ ]:
# قراءة sample مباشرة من SQLite حتى نعرف أسماء الأعمدة المتوفرة.
with sqlite3.connect(DB_PATH) as conn:
    schema_cols = pd.read_sql_query("PRAGMA table_info(documents)", conn)["name"].tolist()
    text_col = "contents" if "contents" in schema_cols else "raw_text"
    sample = pd.read_sql_query(
        f"SELECT doc_id, title, abstract, {text_col} AS text_content FROM documents LIMIT 5",
        conn,
    )

display(sample)

In [ ]:
from src.storage.document_store import DocumentStore

store = DocumentStore(str(DB_PATH))
ids = sample["doc_id"].astype(str).head(3).tolist()
print("Testing DocumentStore.get_by_ids with ids:", ids)

docs = store.get_by_ids(ids)
print("Returned documents:", len(docs))
for doc in docs:
    print("-", doc["doc_id"], "|", (doc.get("title") or "")[:80])